# 4. LLM-режим для многоканального случая

**Цель:** запустить детерминированный `GraphEVTPipeline` на Kuramoto-ряде с впрыснутым событием и попросить LLM проверить результат — обнаружение, задержку и предполагаемый источник — относительно известной разметки. Как и в `1-d/04`, `is_extreme` и `source_channel` используются только для выбора чистого baseline и последующей оценки; сама метка не передаётся численному детектору. Baseline-выбор здесь oracle-assisted и неприменим к неразмеченным реальным данным.

Для живого запуска задайте `MISTRAL_API_KEY` в окружении или введите его в скрытом prompt. Ключ не записывается в ноутбук или артефакты.

In [1]:
from pathlib import Path
import os
import json
from getpass import getpass
import numpy as np
import pandas as pd
from graph_evt_agent import EVTConfig, GraphConfig, GraphEVTPipeline, InputConfig
from graph_evt_agent.agents import EVTAgentTeam, MistralAPIError, MistralClient

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/generated/kuramoto_propagation_series.csv").exists())
DATA = ROOT / "data/generated/kuramoto_propagation_series.csv"
SEED = 42
df = pd.read_csv(DATA)
channel_names = [column for column in df.columns if column.startswith("channel_")]
x = df[channel_names].to_numpy()
event_mask = df["is_extreme"].to_numpy(dtype=bool)
assert np.isfinite(x).all() and x.ndim == 2
assert event_mask.any(), "Generated dataset must contain a labeled event"

propagation_metadata = json.loads((ROOT / "data/generated/synthetic_datasets_metadata.json").read_text())["kuramoto_propagation"]
event_onset = int(np.flatnonzero(event_mask)[0])
event_end = int(np.flatnonzero(event_mask)[-1])
true_source_name = channel_names[int(propagation_metadata["source_channel"])]

In [2]:
api_key = os.getenv("MISTRAL_API_KEY") or getpass("Mistral API key (input hidden): ").strip()
MISTRAL_READY = False
mistral_client = None

if api_key:
    try:
        mistral_client = MistralClient(
            api_key=api_key,
            model="ministral-3b-2512",
            max_retries=0,
        )
        check_reply = mistral_client.complete(
            "Reply exactly with MISTRAL_OK.",
            "Connection check. Reply exactly with MISTRAL_OK.",
        )
        MISTRAL_READY = bool(check_reply.strip())
        print("Mistral API check passed." if MISTRAL_READY else "Mistral returned an empty response.")
    except MistralAPIError as error:
        error_text = str(error).lower()
        if "429" in error_text or "rate_limit" in error_text or "rate limit" in error_text:
            print("Mistral rate limit (HTTP 429). Check your account usage and rate limits, wait, then rerun this cell.")
        else:
            print(f"Mistral API check failed: {error}")
    except Exception as error:
        print(f"Mistral API check failed ({type(error).__name__}).")
else:
    print("Mistral API check skipped: no key was provided.")

Mistral API check failed: Could not reach the Mistral API after 1 attempts: [SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1028). Check connectivity, the selected model, and Mistral service availability.


In [3]:
# Use the known event onset to keep the EVT baseline clean; oracle-assisted,
# only valid because this synthetic recording's onset and source are known.
BASELINE_SIZE = event_onset
pipeline = GraphEVTPipeline(
    EVTConfig(baseline_size=BASELINE_SIZE, tail_quantile=0.85, alarm_probability=0.97, top_k=3, persistence=1),
    graph=GraphConfig(method="ar", edge_threshold=0.35, random_state=SEED),
    input_config=InputConfig(missing="error", require_regular_time=True),
    verbose=True,
)
numeric_result = pipeline.run_detailed(x)
detection = numeric_result.detection
alarm_index = detection.time_index
detected_in_event = alarm_index is not None and bool(event_mask[alarm_index])
detection_delay = alarm_index - event_onset if detected_in_event else None
estimated_source = None if numeric_result.ranking is None else channel_names[numeric_result.ranking.source]
source_correct = estimated_source == true_source_name
plan = numeric_result.plan

pd.DataFrame([{
    "labeled_event_onset": event_onset,
    "labeled_event_end": event_end,
    "baseline_size": BASELINE_SIZE,
    "detected": detection.detected,
    "alarm_index": alarm_index,
    "detected_in_labeled_event": detected_in_event,
    "detection_delay_samples": detection_delay,
    "true_source": true_source_name,
    "estimated_source": estimated_source,
    "source_correct": source_correct,
    "planner_route": None if plan is None else plan.route,
    "stop_reason": None if plan is None else plan.stop_reason,
}])

Graph-EVT pipeline:   0%|          | 0/5 [00:00<?, ?it/s]

,labeled_event_onset,labeled_event_end,baseline_size,detected,alarm_index,detected_in_labeled_event,detection_delay_samples,true_source,estimated_source,source_correct,planner_route,stop_reason
0,932,1036,932,True,949,True,17,channel_00,channel_00,True,n-d,None


In [4]:
if MISTRAL_READY:
    team = EVTAgentTeam(pipeline, mistral_client)
    try:
        agent_report = team.run(
            x,
            task=(
                "Проверь результат EVT-детекции и локализации на многоканальном ряде "
                f"относительно синтетической разметки: истинное событие начинается в {event_onset}, "
                f"истинный источник {true_source_name}; detected={detection.detected}, "
                f"alarm_index={alarm_index}, delay_samples={detection_delay}, "
                f"estimated_source={estimated_source}, source_correct={source_correct}. "
                "Укажи явно, попало ли срабатывание в размеченное событие и совпал ли источник; "
                "не называй правильный источник доказательством того, что метод локализации "
                "надёжен в общем случае."
            ),
        )
        print(agent_report.final_report)
    except MistralAPIError as error:
        error_text = str(error).lower()
        if "429" in error_text or "rate_limit" in error_text or "rate limit" in error_text:
            print("Mistral rate limit during review. Check account usage/rate limits, wait, then rerun this cell.")
        else:
            print(f"LLM review failed: {error}")
    except Exception as error:
        print(f"LLM review failed ({type(error).__name__}).")
else:
    print("LLM review skipped; the numeric pipeline result is still available.")

LLM review skipped; the numeric pipeline result is still available.


`EVTAgentTeam` вызывает тот же детерминированный pipeline и получает `run_detailed`-результат вместе с plan/act/observe trace. В отличие от неразмеченного многоканального ряда, здесь есть известная истина — но она используется только для oracle-assisted baseline и последующей проверки, не для самого детектора. Текст LLM остаётся интерпретацией и должен проверяться по `numeric_result`, а не наоборот.